# 🎓 지식 증류(Knowledge Distillation) 실습 코드
# 전체적인 이야기: "수학 선생님(Teacher)이 초등학생(Student)에게 지혜를 전수하는 과정"

#1. Import Packages

In [ ]:
#1. Import Packages
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


Using cuda device


#2. Preprocessing and downloading the dataset
# 📚 교재 준비: CIFAR-10 = 10가지 사물 사진들 (비행기, 자동차, 새, 고양이, 사슴, 개, 개구리, 말, 배, 트럭)
# 비유: 10가지 동물/사물 그림카드 5만장으로 공부하는 것! 🖼️


In [ ]:
#Preprocessing
transforms_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Loading the CIFAR-10 dataset:
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms_cifar)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms_cifar)

100%|██████████| 170M/170M [00:13<00:00, 12.7MB/s]


#3. Torch data loaders

In [ ]:
#Dataloaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

#4. Teacher Model Class
# 👨‍🏫 수학 선생님 만들기: 똑똑하지만 복잡한 선생님
# 비유: 두뇌 용량 매우 크고 복잡, 비싸고 느리지만 매우 정확한 판단

In [ ]:
# Deeper neural network class to be used as teacher:
class DeepNN(nn.Module):
    def __init__(self, num_classes=10):
        super(DeepNN, self).__init__()
        # 🧠 복잡한 뇌구조: 여러 층의 컨볼루션 레이어들
        # 선생님은 세세한 부분까지 다 볼 수 있어요!
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),  # 128개의 뉴런으로 시작
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        # 🎯 최종 판단부: 복잡한 추론 과정
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),  # 큰 용량의 뇌
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)  # 10가지 클래스 판단
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

#5. Student Model Class
# 👶 초등학생 만들기: 단순하지만 빠른 학생
# 비유: 두뇌 용량 작고 단순, 저렴하고 빠르지만 덜 정확


In [ ]:
# Lightweight neural network class to be used as student:
class LightNN(nn.Module):
    def __init__(self, num_classes=10):
        super(LightNN, self).__init__()
        # 🧠 단순한 뇌구조: 적은 수의 레이어들
        # 학생은 핵심만 빠르게 파악해요!
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),  # 16개의 뉴런으로 시작 (선생님의 1/8)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        # 🎯 최종 판단부: 간단한 추론 과정
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),  # 작은 용량의 뇌 (선생님의 1/2)
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)  # 10가지 클래스 판단
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


#6. Training Step
# 📖 일반적인 학습 함수: 혼자서 교과서 보고 공부하는 방식

In [ ]:
def train(model, train_loader, epochs, learning_rate, device):
    criterion = nn.CrossEntropyLoss()  # 정답만 외우는 방식
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            # inputs: 그림카드들 (batch_size만큼)
            # labels: 정답 라벨들 (0~9 숫자로 표현)
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)

            # outputs: 모델의 예측 (10개 클래스에 대한 확률)
            # labels: 실제 정답 (정수 형태)
            loss = criterion(outputs, labels)  # 정답과 얼마나 다른지 계산
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")


#7. Testing Model
# 📝 시험 보는 함수: 학습한 모델이 얼마나 잘하는지 평가

In [ ]:
def test(model, test_loader, device):
    model.to(device)
    model.eval()  # 시험 모드

    correct = 0
    total = 0

    with torch.no_grad():  # 시험 중에는 학습하지 않음
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)  # 가장 확률 높은 클래스 선택

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy


#8. Teacher Training
# 👨‍🏫 선생님 먼저 공부시키기: 선생님이 충분히 똑똑해져야 학생을 가르칠 수 있어요!

In [ ]:
print("🎓 Step 1: 선생님(Teacher) 훈련 중...")
torch.manual_seed(42)
nn_deep = DeepNN(num_classes=10).to(device)
train(nn_deep, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_deep = test(nn_deep, test_loader, device)


🎓 Step 1: 선생님(Teacher) 훈련 중...
Epoch 1/10, Loss: 1.3368932987418016
Epoch 2/10, Loss: 0.881568441305624
Epoch 3/10, Loss: 0.6954160030845487
Epoch 4/10, Loss: 0.553310876383501
Epoch 5/10, Loss: 0.4356966648260346
Epoch 6/10, Loss: 0.33158469344953745
Epoch 7/10, Loss: 0.24404905401074978
Epoch 8/10, Loss: 0.18560021404948684
Epoch 9/10, Loss: 0.152398221480572
Epoch 10/10, Loss: 0.12243296690952138
Test Accuracy: 74.77%


#9. Declaring the light models
# 👶 비교를 위해 두 명의 학생 준비: 한 명은 혼자 공부, 한 명은 선생님과 공부


In [ ]:
# Instantiate the lightweight network:
torch.manual_seed(42)
nn_light = LightNN(num_classes=10).to(device)  # 혼자 공부할 학생

torch.manual_seed(42)
new_nn_light = LightNN(num_classes=10).to(device)  # 선생님과 공부할 학생

# Print the norm of the first layer of the initial lightweight model
print("Norm of 1st layer of nn_light:", torch.norm(nn_light.features[0].weight).item())
print("Norm of 1st layer of new_nn_light:", torch.norm(new_nn_light.features[0].weight).item())


Norm of 1st layer of nn_light: 2.327361822128296
Norm of 1st layer of new_nn_light: 2.327361822128296


#10. Difference between the parameters
# 📊 선생님과 학생의 뇌 용량 비교: 선생님은 백과사전, 학생은 요약집!

In [ ]:
total_params_deep = "{:,}".format(sum(p.numel() for p in nn_deep.parameters()))
print(f"DeepNN parameters: {total_params_deep}")  # 선생님의 뇌 용량
total_params_light = "{:,}".format(sum(p.numel() for p in nn_light.parameters()))
print(f"LightNN parameters: {total_params_light}")  # 학생의 뇌 용량


DeepNN parameters: 1,186,986
LightNN parameters: 267,738


#11. Student Training
# 📚 학생 혼자 공부: 교과서만 보고 정답 암기하는 방식

In [ ]:
print("📚 Step 2: 학생 혼자 공부 중...")
train(nn_light, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_light_ce = test(nn_light, test_loader, device)


📚 Step 2: 학생 혼자 공부 중...
Epoch 1/10, Loss: 1.4616080598758006
Epoch 2/10, Loss: 1.1514360154681194
Epoch 3/10, Loss: 1.0229638974989772
Epoch 4/10, Loss: 0.9231343702282138
Epoch 5/10, Loss: 0.8506443890769159
Epoch 6/10, Loss: 0.7830313538651332
Epoch 7/10, Loss: 0.7202009476359238
Epoch 8/10, Loss: 0.6627309111225635
Epoch 9/10, Loss: 0.6114538572633358
Epoch 10/10, Loss: 0.5613086086404903
Test Accuracy: 70.66%


#12. Comparing Student vs Teacher

In [ ]:
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy: {test_accuracy_light_ce:.2f}%")

# 🔍 지식 증류가 왜 필요한지 설명
print("\n" + "="*50)
print("지식 증류가 필요한 이유:")
print("="*50)

print("""
🔥 기존 방식 (Hard Target)의 한계:
- 트럭 사진을 보고 → 정답: "트럭!" (100% 확신)
- 나머지 클래스들: 모두 0% (관계 정보 손실)

✨ 지식 증류 방식 (Soft Target)의 장점:
- 트럭 사진을 보고 선생님이 말해요:
  🚛 트럭: 80% 확률
  🚗 자동차: 15% 확률 (비슷하니까!)
  ✈️ 비행기: 4% 확률 (둘 다 교통수단)
  🐕 개: 1% 확률 (전혀 안 비슷해)

이런 미묘한 확률 분포가 바로 'Dark Knowledge'(암묵적 지식)입니다!
""")

# Why soft targets (i.e., the full probability distribution output by a teacher model) can be more useful than hard targets (just the correct class label) when training a student model?

print("""
🎯 Soft Target의 핵심 원리:

1. 🧠 관계 학습: 클래스 간의 유사성을 학습
   - "트럭과 자동차는 비슷해"
   - "개와 고양이는 비슷해"

2. 🔍 세밀한 정보: 작은 확률값들도 중요한 정보
   - Cross-entropy는 정답만 중요하게 여김
   - Soft target은 모든 확률값이 의미 있음

3. 🎨 풍부한 학습 신호: 단순 암기 → 개념 이해
   - Hard: "이건 트럭이야!"
   - Soft: "이건 주로 트럭이지만, 자동차 같기도 해"
""")


Teacher accuracy: 74.77%
Student accuracy: 70.66%

지식 증류가 필요한 이유:

🔥 기존 방식 (Hard Target)의 한계:
- 트럭 사진을 보고 → 정답: "트럭!" (100% 확신)
- 나머지 클래스들: 모두 0% (관계 정보 손실)

✨ 지식 증류 방식 (Soft Target)의 장점:
- 트럭 사진을 보고 선생님이 말해요:
  🚛 트럭: 80% 확률
  🚗 자동차: 15% 확률 (비슷하니까!)
  ✈️ 비행기: 4% 확률 (둘 다 교통수단)
  🐕 개: 1% 확률 (전혀 안 비슷해)

이런 미묘한 확률 분포가 바로 'Dark Knowledge'(암묵적 지식)입니다!


🎯 Soft Target의 핵심 원리:

1. 🧠 관계 학습: 클래스 간의 유사성을 학습
   - "트럭과 자동차는 비슷해" 
   - "개와 고양이는 비슷해"

2. 🔍 세밀한 정보: 작은 확률값들도 중요한 정보
   - Cross-entropy는 정답만 중요하게 여김
   - Soft target은 모든 확률값이 의미 있음

3. 🎨 풍부한 학습 신호: 단순 암기 → 개념 이해
   - Hard: "이건 트럭이야!"
   - Soft: "이건 주로 트럭이지만, 자동차 같기도 해"



#13. Teacher-Student Training
# 🎓 진짜 지식 증류: 선생님의 지혜를 학생에게 전수하는 핵심 과정!

In [ ]:
print("\n" + "="*50)
print("🎓 지식 증류 (Knowledge Distillation) 시작!")
print("="*50)

def train_knowledge_distillation(teacher, student, train_loader, epochs, learning_rate, T, soft_target_loss_weight, ce_loss_weight, device):
    """
    🎓 지식 증류 학습 함수

    이 함수는 학생 모델이 두 가지를 동시에 학습하게 합니다:
    1. 📚 실제 정답 (normal training) - 교과서 공부
    2. 👨‍🏫 선생님의 부드러운 예측 (teacher's softened predictions) - 선생님의 경험담

    Parameters:
    - T: 🌡️ Temperature (온도) - 높을수록 부드러운 답변
    - soft_target_loss_weight: 선생님 지식의 비중 (0.25 = 25%)
    - ce_loss_weight: 교과서 정답의 비중 (0.75 = 75%)
    """
    ce_loss = nn.CrossEntropyLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.eval()  # 👨‍🏫 선생님은 이미 훈련 완료 (평가 모드)
    student.train() # 👶 학생은 훈련 모드

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            # 👨‍🏫 선생님의 예측: 이미 훈련된 선생님이 어떻게 생각하는지 듣기
            with torch.no_grad():  # 선생님은 더 이상 학습하지 않음
                teacher_logits = teacher(inputs)  # 선생님의 원시 예측값

            # 👶 학생의 예측: 같은 문제를 학생이 풀어보기
            student_logits = student(inputs)

            # 🌡️ Temperature로 부드럽게 만들기
            """
            T (Temperature) 매개변수의 역할:
            - T=1: "이건 무조건 트럭!" (딱딱한 답변)
            - T=2: "트럭 같긴 한데..." (적당히 부드러운 답변)
            - T=3: "음... 트럭일 수도, 자동차일 수도..." (매우 부드러운 답변)

            높은 온도 = 더 많은 정보 전달 (Dark Knowledge)
            """
            soft_targets = nn.functional.softmax(teacher_logits / T, dim=-1)  # 선생님의 부드러운 답변
            soft_prob = nn.functional.log_softmax(student_logits / T, dim=-1)  # 학생의 부드러운 답변

            # 💡 KL Divergence: 선생님과 학생의 생각이 얼마나 다른가?
            """
            이 손실은 다음을 측정합니다:
            "학생의 확률 분포가 선생님의 확률 분포와 얼마나 다른가?"

            예시:
            선생님: [트럭:0.8, 자동차:0.15, 비행기:0.04, 개:0.01]
            학생:   [트럭:0.6, 자동차:0.25, 비행기:0.10, 개:0.05]
            → KL Divergence가 이 차이를 수치화

            T²을 곱하는 이유: Hinton 논문에서 권장 (온도 보정)
            """
            soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - soft_prob)) / soft_prob.size()[0] * (T**2)

            # 📚 일반적인 정답 손실: 교과서 정답과 비교
            label_loss = ce_loss(student_logits, labels)

            # ⚖️ 두 지식을 적절히 섞기: 선생님 경험담 25% + 교과서 정답 75%
            """
            최종 손실 함수의 의미:
            - 0.25 * soft_targets_loss: 선생님의 미묘한 지혜 (25%)
            - 0.75 * label_loss: 확실한 정답 지식 (75%)

            이 비율은 조정 가능:
            - 선생님을 더 믿는다면 → soft_target_loss_weight ↑
            - 정답을 더 믿는다면 → ce_loss_weight ↑
            """
            loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss

            # 🔄 역전파: 계산된 손실을 바탕으로 학생의 뇌 업데이트
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

# 🎓 실제 지식 증류 실행!
print("🎓 Step 3: 선생님과 함께 공부하는 학생 훈련 중...")
print("📊 설정: Temperature=2, 선생님 지식 75% + 정답 25%")

# Apply knowledge distillation with temperature of 2
# 가중치 설정: 0.25 for distillation loss, 0.75 for CE loss
train_knowledge_distillation(
    teacher=nn_deep,
    student=new_nn_light,
    train_loader=train_loader,
    epochs=10,
    learning_rate=0.001,
    T=2,  # 🌡️ 온도: 적당히 부드러운 답변
    soft_target_loss_weight=0.25,  # 선생님 지식 25%
    ce_loss_weight=0.75,  # 정답 지식 75%
    device=device
)

test_accuracy_light_ce_and_kd = test(new_nn_light, test_loader, device)

# 🏆 최종 결과 비교: 지식 증류의 효과 확인!
print("\n" + "="*50)
print("🏆 최종 성능 비교 결과")
print("="*50)

print(f"👨‍🏫 Teacher accuracy (선생님): {test_accuracy_deep:.2f}%")
print(f"👶 Student accuracy without teacher (혼자 공부한 학생): {test_accuracy_light_ce:.2f}%")
print(f"🎓 Student accuracy with KD (선생님과 공부한 학생): {test_accuracy_light_ce_and_kd:.2f}%")

print(f"\n✨ 지식 증류 효과: +{test_accuracy_light_ce_and_kd - test_accuracy_light_ce:.2f}%p 향상!")

print("""
🎉 지식 증류의 마법:
- 📖 혼자 공부: 단순 암기 (Hard Label)
- 🎓 선생님과 공부: 관계 이해 + 미묘한 패턴 학습 (Soft Label)
- 🌟 결과: 작은 모델이지만 더 똑똑해짐!

💡 핵심 아이디어:
"선생님의 확률 분포 = Dark Knowledge (암묵적 지식)"
이 미묘한 정보들이 학생을 더 똑똑하게 만들어 줍니다!
""")


🎓 지식 증류 (Knowledge Distillation) 시작!
🎓 Step 3: 선생님과 함께 공부하는 학생 훈련 중...
📊 설정: Temperature=3, 선생님 지식 75% + 정답 25%
Epoch 1/10, Loss: 0.35245840640171716
Epoch 2/10, Loss: 0.29997393599404093
Epoch 3/10, Loss: 0.28007254054021957
Epoch 4/10, Loss: 0.2704026293571648
Epoch 5/10, Loss: 0.2579177178613975
Epoch 6/10, Loss: 0.252212607151712
Epoch 7/10, Loss: 0.24484898835954155
Epoch 8/10, Loss: 0.2311085111573529
Epoch 9/10, Loss: 0.23328073581923608
Epoch 10/10, Loss: 0.2297618153226345
Test Accuracy: 70.46%

🏆 최종 성능 비교 결과
👨‍🏫 Teacher accuracy (선생님): 74.77%
👶 Student accuracy without teacher (혼자 공부한 학생): 70.66%
🎓 Student accuracy with KD (선생님과 공부한 학생): 70.46%

✨ 지식 증류 효과: +-0.20%p 향상!

🎉 지식 증류의 마법:
- 📖 혼자 공부: 단순 암기 (Hard Label)
- 🎓 선생님과 공부: 관계 이해 + 미묘한 패턴 학습 (Soft Label)
- 🌟 결과: 작은 모델이지만 더 똑똑해짐!

💡 핵심 아이디어:
"선생님의 확률 분포 = Dark Knowledge (암묵적 지식)"
이 미묘한 정보들이 학생을 더 똑똑하게 만들어 줍니다!

